In [6]:
from pymongo import MongoClient
from geopy.distance import geodesic
from decimal import Decimal
import random
import datetime
import pandas as pd
import pydeck as pdk

client = MongoClient("mongodb://localhost:27017/")
db = client['delivery']
orders_collection = db['orders']
drivers_collection = db['drivers']

def generate_random_coordinate():
    lat = round(random.uniform(12.80, 13.10), 6)
    lon = round(random.uniform(77.40, 77.80), 6)
    return (lat, lon)

def generate_random_person(details):
    return {
        "name": f"{details.capitalize()} {random.randint(1, 100)}",
        "phone": f"+91-9{random.randint(100000000, 999999999)}",
        "address": f"D.No:{random.randint(1, 500)}, Bangalore"
    }

#Calculate Cost and Earnings
def calculate_cost_and_earnings(distance):
    cost = Decimal(30) + Decimal(distance * 10)
    fare_earnings = Decimal(0.8) * cost
    return cost, fare_earnings
    
drivers = []
driver_map = {}
for i in range(5):
    current_location = generate_random_coordinate()
    driver = {
        "driver_id": f"DRV{i+1:03}",
        "name": f"Driver{i+1}",
        "phone": f"+91-900000000{i}",
        "vehicle_number": f"KA01AB{1000+i}",
        "current_location": {
            "latitude": current_location[0],
            "longitude": current_location[1]
        },
        "availability": random.choice(["yes", "no"]),
        "total_distance_km": 0,
        "total_earnings": 0
    }
    drivers.append(driver)
    driver_map[driver["driver_id"]] = driver

def generate_orders(drivers, num_orders=25):
    orders=[]
    delivery_km = {driver["driver_id"]: Decimal(0) for driver in drivers}
    for i in range(num_orders):
        pickup_location = generate_random_coordinate()
        drop_location = generate_random_coordinate()
        distance = Decimal(round(geodesic(pickup_location, drop_location).kilometers, 2))

        status = random.choice(["pending", "in_transit", "delivered"])
        pickup_time = None
        drop_time = None
        
        if status == "delivered":
            pickup_time = datetime.datetime.now() - datetime.timedelta(days=random.randint(1, 7))
            drop_time = pickup_time + datetime.timedelta(minutes=float(distance)*2)
        elif status == "in_transit":
            pickup_time = datetime.datetime.now() - datetime.timedelta(hours=random.randint(1, 5))
        
        driver = random.choice(drivers)
        driver_id = driver["driver_id"]

        cost, fare_earnings = calculate_cost_and_earnings(distance)

        projected_total_distance = Decimal(driver["total_distance_km"]) + distance
        projected_delivery_distance = delivery_km[driver_id] + distance
        extra_km = max(Decimal(0), projected_total_distance - projected_delivery_distance)
        extra_earnings = extra_km * Decimal(5)

        earnings = fare_earnings + extra_earnings
        
        driver["total_distance_km"] += float(distance)
        delivery_km[driver_id] += distance
        driver["total_earnings"] += earnings

        order = {
            "order_id": f"ORD{i+1:05}",
            "pickup_coordinates": {
                "latitude": pickup_location[0],
                "longitude": pickup_location[1]
            },
            "drop_coordinates": {
                "latitude": drop_location[0],
                "longitude": drop_location[1]
            },
            "pickup_time": pickup_time.isoformat() if pickup_time else None,
            "drop_time": drop_time.isoformat() if drop_time else None,
            "status": status,
            "driver_id": driver_id,
            "distance_covered_km": float(distance),
            "cost": float(cost),
            "sender": generate_random_person("sender"),
            "receiver": generate_random_person("receiver"),
            "weight_kg": round(random.uniform(0.1, 20), 2)
        }
        orders.append(order)
    return orders

orders = generate_orders(drivers, 25)

for driver in drivers:
    driver['total_earnings'] = float(driver['total_earnings'])
    driver['total_distance_km'] = float(driver['total_distance_km'])

drivers_collection.delete_many({})
orders_collection.delete_many({})
drivers_collection.insert_many(drivers)
orders_collection.insert_many(orders)

orders_collection.create_index("order_id", unique=True)
orders_collection.create_index("driver_id")
orders_collection.create_index("cost")
orders_collection.create_index("sender.phone")
orders_collection.create_index("receiver.phone")
orders_collection.create_index("weight_kg")
drivers_collection.create_index("driver_id", unique=True)
drivers_collection.create_index("name")
drivers_collection.create_index("phone")

orders = list(orders_collection.find())

pickup_points = []
drop_points = []
lines = []

for order in orders:
    pickup = order["pickup_coordinates"]
    drop = order["drop_coordinates"]

    pickup_points.append({
        "lat": pickup["latitude"],
        "lon": pickup["longitude"],
        "type": "Pickup",
        "order_id": order["order_id"]
    })

    drop_points.append({
        "lat": drop["latitude"],
        "lon": drop["longitude"],
        "type": "Drop",
        "order_id": order["order_id"]
    })

    lines.append({
        "from_lon": pickup["longitude"],
        "from_lat": pickup["latitude"],
        "to_lon": drop["longitude"],
        "to_lat": drop["latitude"]
    })

pickup_df = pd.DataFrame(pickup_points)
drop_df = pd.DataFrame(drop_points)
lines_df = pd.DataFrame(lines)

pdk.settings.mapbox_api_key = "pk.eyJ1IjoibW9rc2hpdGgxMCIsImEiOiJjbWJieGY5cTUwbGFzMnNzYXlnNTIyeGdlIn0.R_VR8JHIshOwq-5Gymsc7A"

pickup_layer = pdk.Layer(
    "ScatterplotLayer",
    data=pickup_df,
    get_position='[lon, lat]',
    get_fill_color='[255, 0, 0]',
    get_radius=50,
    pickable=True,
    tooltip=True
)

drop_layer = pdk.Layer(
    "ScatterplotLayer",
    data=drop_df,
    get_position='[lon, lat]',
    get_fill_color='[0, 200, 0]',
    get_radius=50,
    pickable=True,
    tooltip=True
)

line_layer = pdk.Layer(
    "LineLayer",
    data=lines_df,
    get_source_position='[from_lon, from_lat]',
    get_target_position='[to_lon, to_lat]',
    get_width=2,
    get_color=[0, 120, 255],
    pickable=False
)

view_state = pdk.ViewState(
    latitude=12.9716,
    longitude=77.5946,
    zoom=10,
    pitch=0
)

deck = pdk.Deck(
    layers=[pickup_layer, drop_layer, line_layer],
    initial_view_state=view_state,
    tooltip={"text": "{type} - {order_id}"}
)

deck = pdk.Deck(
    layers = [pickup_layer, drop_layer, line_layer],
    initial_view_state=view_state,
    tooltip={"text": "{type} - {order_id}"}
)
deck.show()